# EDA On `games.csv` Dataset


In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load Data
# ==========================================

games_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "games.csv"), 
    )
)

print("Games Data:")
print(f"    - Dimensions: {games_df.shape}")
print(f"    - Number of Unique Games: {games_df.select("game_key").n_unique()}")
games_df

In [ ]:
games_df.filter(pl.col("game_id") == 2023101507)

In [ ]:
# =======================================
# Missing Data
# =======================================

# No missing data

games_df.select(pl.all().is_null().sum())

In [ ]:
# ==================================
# game_id / game_key
# ==================================

print(f"  - Number of Unique game_id values: {games_df.select("game_id").n_unique()}")
print(f"  - Number of Unique game_key values: {games_df.select("game_key").n_unique()}")

In [ ]:
# ==================================
# season
# ==================================

games_df.group_by("season").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count"))

In [ ]:
# ==================================
# season_type
# ==================================

games_df.group_by("season_type").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==================================
# week
# ==================================
print(f"Number of Unique Weeks: {games_df.select("week").n_unique()}")
print(sorted(games_df.select("week").unique().to_series().to_list()))

In [ ]:
# ==================================
# season_type / week
# ==================================

games_df_with_stw = (
    games_df
    .with_columns(season_type_week = pl.col("season_type").cast(pl.String) + "_" + pl.col("week").cast(pl.String))
)

print(f"Number of Unique Season-Type/Weeks: {games_df_with_stw.select("season_type_week").n_unique()}")
print(sorted(games_df_with_stw.select("season_type_week").unique().to_series().to_list()))

In [ ]:
# ==================================
# home_team_abbr
# ==================================

# NOTE NOTE NOTE: Sometimes the Rams (LAR) and Chargers (LAC) are put down as "LA"

print(f"Number of Home Teams (not missing): {games_df.filter(pl.col("home_team_abbr").is_not_null()).select("home_team_abbr").n_unique()}")
print(sorted(games_df.filter(pl.col("home_team_abbr").is_not_null()).select("home_team_abbr").unique().to_series().to_list()))

In [ ]:
# ==================================
# visitor_team_abbr
# ==================================

# NOTE NOTE NOTE: Sometimes the Rams (LAR) and Chargers (LAC) are put down as "LA"

print(f"Number of Away Teams (not missing): {games_df.filter(pl.col("visitor_team_abbr").is_not_null()).select("visitor_team_abbr").n_unique()}")
print(sorted(games_df.filter(pl.col("visitor_team_abbr").is_not_null()).select("visitor_team_abbr").unique().to_series().to_list()))